# Lab: LangSmith 셋업 및 Structured Output 실습 (WE DO)

PydanticAI Agent에서 LangSmith 트레이싱이 동작하는 전체 흐름을 실습한다.

- 환경변수 확인
- `wrap_openai`로 OpenAI 클라이언트 트레이싱
- PydanticAI Agent + LangSmith 트레이싱
- Structured Output: Pydantic 모델 정의 -> Agent로 구조화된 응답 받기

**사전 준비**: `.env` 파일에 다음 설정
```
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=day1-context-strategy
```

## 환경 설정

`.env` 파일에서 API 키와 LangSmith 환경변수를 로드한다.
`load_dotenv()`는 현재 디렉토리의 `.env` 파일을 찾아 `os.environ`에 반영한다.

In [3]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경변수를 로드 (OPENAI_API_KEY, LANGSMITH_API_KEY 등)
load_dotenv()

True

---
## Step 1: 환경변수 확인

LangSmith 트레이싱에 필요한 환경변수가 제대로 설정되었는지 확인한다.
하나라도 빠져 있으면 트레이스가 기록되지 않으므로 반드시 확인이 필요하다.

| 환경변수 | 역할 |
|---------|------|
| `OPENAI_API_KEY` | OpenAI API 호출 인증 |
| `LANGSMITH_API_KEY` | LangSmith에 트레이스 전송 인증 |
| `LANGSMITH_TRACING` | `true`로 설정해야 트레이싱 활성화 |
| `LANGSMITH_PROJECT` | 트레이스를 기록할 프로젝트 이름 (선택) |

In [4]:
# 필수 환경변수 목록과 설명
required = {
    "OPENAI_API_KEY": "OpenAI API 키",
    "LANGSMITH_API_KEY": "LangSmith API 키 (https://smith.langchain.com 에서 발급)",
    "LANGSMITH_TRACING": "트레이싱 활성화 (true로 설정)",
}

all_ok = True
for key, desc in required.items():
    value = os.environ.get(key)
    if value:
        # API 키 보안을 위해 앞 8자만 표시
        masked = value[:8] + "..." if len(value) > 8 else value
        print(f"[OK] {key} = {masked}")
    else:
        print(f"[!!] {key} 미설정 - {desc}")
        all_ok = False

# 선택: 프로젝트 이름 (미설정 시 'default' 프로젝트에 기록)
project = os.environ.get("LANGSMITH_PROJECT", "(미설정 - 기본값 'default' 사용)")
print(f"[..] LANGSMITH_PROJECT = {project}")

if not all_ok:
    print("\n.env 파일 예시:")
    print("OPENAI_API_KEY=sk-...")
    print("LANGSMITH_API_KEY=lsv2_...")
    print("LANGSMITH_TRACING=true")
    print("LANGSMITH_PROJECT=day1-context-strategy")

[OK] OPENAI_API_KEY = sk-proj-...
[OK] LANGSMITH_API_KEY = lsv2_pt_...
[OK] LANGSMITH_TRACING = true
[..] LANGSMITH_PROJECT = day1-labs


---
## Step 2: wrap_openai - OpenAI 클라이언트 트레이싱

`wrap_openai`로 OpenAI 클라이언트를 감싸면 모든 API 호출이 LangSmith에 자동으로 기록된다.

**동작 원리:**
1. `wrap_openai(OpenAI())`가 원본 클라이언트를 프록시로 감싼다
2. 이후 이 클라이언트로 하는 모든 `chat.completions.create()` 호출이 LangSmith에 트레이스로 전송된다
3. 입력 프롬프트, 출력 응답, 토큰 수, 지연 시간 등이 자동 기록된다

PydanticAI가 내부적으로 OpenAI 클라이언트를 사용하므로,
이 패턴을 먼저 익힌 후 PydanticAI Agent에 적용한다.

In [5]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai

# wrap_openai로 클라이언트를 감싼다
# 이후 이 client로 호출하는 모든 API가 LangSmith에 기록된다
client = wrap_openai(OpenAI())

print("OpenAI 클라이언트를 wrap_openai로 래핑 완료")
print("이제 모든 API 호출이 LangSmith에 자동 기록됩니다")

OpenAI 클라이언트를 wrap_openai로 래핑 완료
이제 모든 API 호출이 LangSmith에 자동 기록됩니다


In [8]:
# wrap_openai로 감싼 클라이언트로 간단한 호출 테스트
response = client.chat.completions.create(
    model="gpt-5.4",
    temperature=0,  # 재현 가능하도록 온도를 0으로 설정
    messages=[
        {"role": "system", "content": "한국어로 답하라."},
        {"role": "user", "content": "LangSmith가 뭔지 한 문장으로 설명해줘."},
    ],
    max_completion_tokens=100,  # gpt-5.4는 max_tokens 대신 max_completion_tokens 사용
)

print(f"응답: {response.choices[0].message.content}")
print(f"토큰: {response.usage.prompt_tokens} (입력) + {response.usage.completion_tokens} (출력)")
print(f"\nLangSmith 대시보드에서 이 호출의 트레이스를 확인하세요:")
print("https://smith.langchain.com")

응답: LangSmith는 LLM 애플리케이션의 디버깅, 평가, 모니터링을 돕는 개발자용 플랫폼입니다.
토큰: 31 (입력) + 35 (출력)

LangSmith 대시보드에서 이 호출의 트레이스를 확인하세요:
https://smith.langchain.com


---
## Step 3: PydanticAI Agent + LangSmith 트레이싱

PydanticAI Agent도 내부적으로 OpenAI 클라이언트를 사용한다.
`LANGSMITH_TRACING=true` 환경변수만 설정되어 있으면 자동으로 트레이싱이 동작한다.

> **Jupyter 노트북에서의 주의점**: Jupyter는 이미 이벤트 루프가 실행 중이므로
> `agent.run_sync()`를 호출하면 `RuntimeError: This event loop is already running` 오류가 발생한다.
> 대신 `await agent.run()`을 사용한다. Jupyter는 top-level `await`를 지원한다.

In [5]:
from pydantic_ai import Agent

# PydanticAI Agent 생성 — instructions로 분류 역할 부여
agent = Agent(
    "openai:gpt-5.4",
    instructions="고객 리뷰의 감성을 한 단어로 분류하라: 긍정, 부정, 중립.",
)

# 다양한 감성의 테스트 리뷰
reviews = [
    "완전 최고에요! 강추합니다!",
    "배송은 빨랐는데 품질이 기대 이하네요.",
    "그냥 그래요. 특별한 건 없어요.",
]

for review in reviews:
    # await agent.run() 사용 — Jupyter에서는 run_sync() 대신 await를 쓴다
    result = await agent.run(f"리뷰: {review}")
    usage = result.usage()
    print(f"리뷰: \"{review}\"")
    print(f"  감성: {result.output}")
    print(f"  토큰: {usage.input_tokens} (입력) + {usage.output_tokens} (출력)")
    print()

print("LangSmith 대시보드에서 PydanticAI 트레이스를 확인하세요")
print("각 await agent.run() 호출이 개별 트레이스로 기록됩니다")

리뷰: "완전 최고에요! 강추합니다!"
  감성: 긍정
  토큰: 48 (입력) + 6 (출력)

리뷰: "배송은 빨랐는데 품질이 기대 이하네요."
  감성: 부정
  토큰: 51 (입력) + 5 (출력)

리뷰: "그냥 그래요. 특별한 건 없어요."
  감성: 중립
  토큰: 48 (입력) + 5 (출력)

LangSmith 대시보드에서 PydanticAI 트레이스를 확인하세요
각 await agent.run() 호출이 개별 트레이스로 기록됩니다


---
## Step 4: Structured Output 실습

Pydantic 모델을 정의하고 PydanticAI Agent의 `output_type`으로 전달하면
LLM 응답이 해당 스키마에 맞게 100% 강제된다.

**핵심 개념:**
- `output_type`에 Pydantic 모델을 지정 → LLM이 JSON Schema를 참조하여 응답
- `result.output`이 Pydantic 모델 인스턴스 → `.sentiment`, `.keywords` 등 타입 안전 접근
- `Field(description=...)`은 LLM에게 각 필드의 의미를 알려주는 힌트

In [6]:
from pydantic import BaseModel, Field
from enum import Enum


# Enum으로 감성 값을 3가지로 제한 — LLM이 범위 밖의 값을 생성할 수 없다
class Sentiment(str, Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"


# Pydantic 모델로 응답 스키마를 정의
# Field의 description은 LLM이 각 필드를 이해하는 힌트
class ReviewAnalysis(BaseModel):
    """리뷰 분석 결과"""
    sentiment: Sentiment = Field(description="감성 분류")
    confidence: float = Field(description="신뢰도 (0.0~1.0)", ge=0, le=1)
    keywords: list[str] = Field(description="핵심 키워드 (최대 3개)")
    summary: str = Field(description="리뷰 요약 (1문장)")

In [7]:
# output_type에 Pydantic 모델을 지정 — 응답이 ReviewAnalysis 스키마에 강제된다
agent = Agent(
    "openai:gpt-5.4",
    instructions="고객 리뷰를 분석하여 구조화된 결과를 반환하세요.",
    output_type=ReviewAnalysis,
)

# 긍정/부정이 뚜렷한 테스트 리뷰
reviews = [
    "가격은 좀 비싸지만 품질이 정말 좋아요. 재구매 의사 있습니다.",
    "한 달 쓰다가 고장났어요. AS도 느리고 실망입니다.",
]

for review in reviews:
    # await agent.run() — Jupyter 환경에서의 비동기 호출
    result = await agent.run(f"리뷰: {review}")
    output = result.output  # ReviewAnalysis 인스턴스
    usage = result.usage()

    print(f"리뷰: \"{review}\"")
    # Pydantic 모델 필드에 타입 안전하게 접근
    print(f"  감성:   {output.sentiment.value} (신뢰도: {output.confidence:.0%})")
    print(f"  키워드: {', '.join(output.keywords)}")
    print(f"  요약:   {output.summary}")
    print(f"  토큰:   {usage.input_tokens} (입력) + {usage.output_tokens} (출력)")
    print()

리뷰: "가격은 좀 비싸지만 품질이 정말 좋아요. 재구매 의사 있습니다."
  감성:   긍정 (신뢰도: 90%)
  키워드: 가격, 품질, 재구매
  요약:   가격은 다소 비싸지만 품질에 매우 만족하며 재구매 의사가 있는 긍정적인 리뷰입니다.
  토큰:   244 (입력) + 67 (출력)

리뷰: "한 달 쓰다가 고장났어요. AS도 느리고 실망입니다."
  감성:   부정 (신뢰도: 98%)
  키워드: 고장, AS 지연, 실망
  요약:   한 달 사용 후 제품이 고장났고 AS 처리도 느려 전반적으로 매우 실망한 리뷰입니다.
  토큰:   241 (입력) + 66 (출력)



### 핵심 포인트

- `output_type=ReviewAnalysis`로 스키마를 강제
- `result.output`이 `ReviewAnalysis` 타입 -> 타입 안전한 접근
- `.sentiment`, `.keywords`, `.summary` 등 필드에 직접 접근 가능
- LangSmith에서 스키마 정보와 함께 트레이스 확인 가능

---
## 실습 완료!

**LangSmith 대시보드 확인 사항:**

1. https://smith.langchain.com 접속
2. 프로젝트 `day1-context-strategy` 선택
3. 트레이스 목록에서 각 호출의 입력/출력 확인
4. `wrap_openai` 트레이스 vs PydanticAI 트레이스 비교
5. Structured Output 호출의 스키마 정보 확인
6. 토큰 사용량, 지연 시간 등 메타데이터 확인

**PydanticAI 핵심 패턴 요약:**

```python
# Agent 생성
agent = Agent(model, instructions=..., output_type=...)

# Jupyter에서 실행 (await 사용)
result = await agent.run(prompt)

# 결과 접근
result.output      # Pydantic 모델 인스턴스 (output_type 지정 시)
result.usage()     # RunUsage(input_tokens, output_tokens, requests)
```

> **주의**: 일반 Python 스크립트에서는 `agent.run_sync(prompt)`를 사용해도 되지만,
> Jupyter 노트북에서는 반드시 `await agent.run(prompt)`를 사용해야 한다.